In [1]:
# Pass marks, set before the run.
# The reproduction counts as matching the paper if every category is within this of its printed score.
MATCH_WITHIN = 0.03
# Our recomputation of the paper's averages from the uploaded predictions must agree with the trainer's
# own table to within this, or the scoring here is not the paper's.
RECOMPUTE_WITHIN = 0.02
# The runs, by HumSet version: 2.0.0 is the version the paper's code loads; 1.0.0 is this repository's.
RUNS = {"2.0.0": "full", "1.0.0": "full_v1"}

# 13 · HumBERT's HumSet result, reproduced with its authors' trainer

The published HumBERT figures (Tamagnone et al., IJCAI 2023, arXiv 2305.16756) are the bar our combined method
misses on HumSet. Reading the authors' code showed two things the paper does not say: it loads HumSet with no
version, which since February 2023 means version 2.0.0, a different test set from the version 1.0.0 we use; and
it averages subpillars within each pillar before averaging pillars.

`scripts/hf_jobs/reproduce_humbert.py` ran the authors' packaged trainer, `the-deep-nlp/classifier-trainer` at a
fixed commit, unchanged, with the paper notebook's settings, once on each version's own splits, on one A10G each.
This notebook reads what the runs uploaded and asks two questions. On version 2.0.0: does the printed result
reproduce? On version 1.0.0: how does the paper's recipe do on the test set our methods are scored on?

Reductions: one run and one seed per version; the packaged trainer is the authors' September 2023 packaging of
the code, not the May 2023 notebook itself; library versions are the newest that run it (the launcher lists
them).

In [2]:
import csv
import json

import pandas as pd

from tiered_taxonomy import benchmarks as bm
from tiered_taxonomy import experiment as ex
from tiered_taxonomy import gpu_jobs as gj
from tiered_taxonomy import humbert_reproduction as hr
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root

root = data_root()


def fetch(run, name):
    return gj.fetch(root, f"humbert_reproduction/{run}/{name}")


def lines(run, name):
    return [json.loads(line) for line in open(fetch(run, name), encoding="utf-8")]


def scored(run):
    info = json.loads(open(fetch(run, "run.json"), encoding="utf-8").read())
    if info["sample"]:
        raise ValueError(f"{run} is a smoke test, not a full run")
    thresholds = json.loads(open(fetch(run, "thresholds.json"), encoding="utf-8").read())
    table = hr.table_scores(list(csv.DictReader(open(fetch(run, "results.csv"), encoding="utf-8"))))
    f1 = hr.label_f1(lines(run, "predictions.jsonl"), thresholds)
    nested, flat = hr.category_scores(f1, nested=True), hr.category_scores(f1, nested=False)
    gaps = {name: abs(nested[name] - table[name]) for name in hr.CATEGORIES}
    if max(gaps.values()) > RECOMPUTE_WITHIN:
        raise ValueError(f"{run}: the recomputation strays from the trainer's table: {gaps}")
    table["Mean"] = sum(table[name] for name in hr.CATEGORIES) / len(hr.CATEGORIES)
    return {"run": info, "trainer_table": table, "paper_averages": nested, "every_label_once": flat}


results = {version: scored(run) for version, run in RUNS.items()}
for version, entry in results.items():
    info = entry["run"]
    print(version, info["sizes"], info["device"], f"{info['hours']} hours")

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.0.0 {'train': 129268, 'validation': 11940, 'test': 12896} NVIDIA A10G 1.13 hours
1.0.0 {'train': 117435, 'validation': 16039, 'test': 15147} NVIDIA A10G 1.086 hours


## On version 2.0.0, the paper's data: does the printed result reproduce?

The first column is the trainer's results table, the paper's own scoring. The second averages every label once,
as our other HumSet scores do. The trainer's table is recomputed from the uploaded predictions and checked against
itself first.

In [3]:
printed = {**bm.HUMBERT_BEST, "Mean": sum(bm.HUMBERT_BEST.values()) / len(bm.HUMBERT_BEST)}
paper = results["2.0.0"]
print(
    "trainer's Avg (mean of the three tier-1 categories):",
    round(paper["trainer_table"]["Avg"], 3),
    "printed: 0.73",
)
matches = {name: abs(paper["trainer_table"][name] - printed[name]) <= MATCH_WITHIN for name in hr.CATEGORIES}
results["2.0.0"]["matches_printed"] = matches
results["2.0.0"]["reproduces"] = all(matches.values())
print("reproduces the printed score within", MATCH_WITHIN, ":", matches)
pd.DataFrame(
    {
        "trainer's table": paper["trainer_table"],
        "every label once": paper["every_label_once"],
        "HumBERT, printed": printed,
    }
).loc[[*hr.CATEGORIES, "Mean"]].round(3)

trainer's Avg (mean of the three tier-1 categories): 0.724 printed: 0.73
reproduces the printed score within 0.03 : {'Sectors': True, 'Pillars 1D': True, 'Subpillars 1D': False, 'Pillars 2D': True, 'Subpillars 2D': True}


,trainer's table,every label once,"HumBERT, printed"
Sectors,0.779,0.778,0.780
Pillars 1D,0.688,0.685,0.700
Subpillars 1D,0.499,0.548,0.540
Pillars 2D,0.704,0.705,0.710
Subpillars 2D,0.485,0.501,0.510
Mean,0.631,0.643,0.648


## On version 1.0.0, our test set: the paper's recipe beside our methods

Every label is averaged once here, as in notebook 10, which scored our combined method on the same 15,147 test
excerpts. The two choose their thresholds differently: the trainer on all 16,039 development excerpts, notebook 10
on its 4,000 cut items.

In [4]:
ours = ex.load_results("10_combined_method")["humset"]
recipe = results["1.0.0"]
pd.DataFrame(
    {
        "HumBERT recipe, trainer's table": recipe["trainer_table"],
        "HumBERT recipe, every label once": recipe["every_label_once"],
        "our combined method": ours["scores"],
        "our fine-tuned HumBERT": ours["best_single"],
        "HumBERT, printed (on 2.0.0)": printed,
    }
).loc[[*hr.CATEGORIES, "Mean"]].round(3)

,"HumBERT recipe, trainer's table","HumBERT recipe, every label once",our combined method,our fine-tuned HumBERT,"HumBERT, printed (on 2.0.0)"
Sectors,0.734,0.734,0.742,0.726,0.780
Pillars 1D,0.533,0.532,0.553,0.549,0.700
Subpillars 1D,0.380,0.428,0.453,0.434,0.540
Pillars 2D,0.642,0.640,0.646,0.633,0.710
Subpillars 2D,0.447,0.463,0.475,0.461,0.510
Mean,0.547,0.560,0.574,0.561,0.648


## The recipe's rankings, saved as a source

Its label probabilities on every development and test excerpt are saved as the source `humbert_recipe`, in this
repository's label ids, so that notebook 14 can score it on the same items and the same way as every other
method.

In [5]:
humset = ex.load_humset(root)
for split, items, name in (
    ("dev", humset.dev, "rankings_validation.jsonl"),
    ("test", humset.test, "rankings_test.jsonl"),
):
    rankings = hr.to_rankings(lines(RUNS["1.0.0"], name))
    missing = {item.id for item in items} - set(rankings)
    if missing:
        raise ValueError(f"{split}: {len(missing)} excerpts have no ranking, for example {min(missing)}")
    sg.save_signal(root, humset.name, "humbert_recipe", split, rankings)
    print(split, len(rankings), "rankings saved")

dev 16034 rankings saved


test 15142 rankings saved


In [6]:
ex.save_results("13_humbert_reproduction", results)
print("saved")

saved
